# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [1]:
!pip install chromadb langchain langchain_community langchain_openai langchain_text_splitters langchain-classic

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_classic.chains import RetrievalQA  # Legacy chains
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader


from langchain_classic.agents import AgentType, initialize_agent
from langchain_openai import ChatOpenAI

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [4]:
# If you're using colab, run this
#os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [6]:
from pathlib import Path

doc_path = str(Path(".") / "sonnets.txt")

In [7]:
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="sonnets")

In [8]:
sonnets_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [9]:
import os
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"

In [10]:
from langchain_community.document_loaders import WebBaseLoader

In [11]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [12]:
%pip install beautifulsoup4 lxml html5lib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Put near where bs4 is imported
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "bs4 (beautifulsoup4) is required. Install with `pip install beautifulsoup4`"
    ) from e

In [14]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1256, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


## Create the Agent

In [15]:
# Import things that are needed generically
# Legacy agent imports
from langchain_classic.agents import AgentType, initialize_agent
from langchain_classic.tools import Tool  # or langchain.tools.Tool if using v1 Tool
from langchain_openai import OpenAI

In [16]:
tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description="useful for when you need to answer questions about Shakespeare's sonnets. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [17]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\sisaz\AppData\Local\Temp\ipykernel_43388\1834837320.py:3: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(


In [18]:
agent.invoke("What is the main theme of Shakespeare's Sonnet 18?")



> Entering new AgentExecutor chain...
I need to determine the main theme of Shakespeare's Sonnet 18, which is often referred to as "Shall I compare thee to a summer's day?" 
Action: Shakespeare Sonnets QA System
Action Input: What is the main theme of Shakespeare's Sonnet 18?
Observation: The main theme of Shakespeare's Sonnet 18 is the enduring nature of beauty and love. The speaker compares the beloved to a summer's day, highlighting that while summer is temporary and subject to change, the beauty of the beloved will not fade. The poem suggests that through poetry, the beloved's beauty can be immortalized and preserved against the ravages of time and death.
Thought:I now know the final answer.  
Final Answer: The main theme of Shakespeare's Sonnet 18 is the enduring nature of beauty and love, emphasizing that while summer is temporary, the beloved's beauty can be immortalized through poetry.

> Finished chain.


{'input': "What is the main theme of Shakespeare's Sonnet 18?",
 'output': "The main theme of Shakespeare's Sonnet 18 is the enduring nature of beauty and love, emphasizing that while summer is temporary, the beloved's beauty can be immortalized through poetry."}

In [19]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to compare the features and benefits of ruff and flake8 to provide a comprehensive answer about why one might choose ruff over flake8. 
Action: Ruff QA System
Action Input: Why should someone use ruff instead of flake8?
Observation: Someone might choose to use Ruff instead of Flake8 for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 800 rules, including many popular Flake8 plugins natively, which can reduce the need for multiple plugins.

3. **Automatic Fixing**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, but it re-implements popular Flake8 plugins, which can simplify the setup.

5. **Compatibility with Black**: Ruff's linter is compatible with Black out-of-the-box, making it easier to integ

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for several reasons, including its faster performance, comprehensive rule set with over 800 rules, automatic fixing of lint violations, compatibility with Black, and simplified configuration. Ruff can also serve as a drop-in replacement for Flake8, making it easier for users to switch.'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [20]:
tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description="useful for when you need to answer questions about Shakespeare's sonnets. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [21]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [22]:
agent.invoke("How does Shakespeare describe love and time in his sonnets?")



> Entering new AgentExecutor chain...
I need to gather information about how Shakespeare portrays the themes of love and time in his sonnets. This will likely involve analyzing specific sonnets that address these themes. 
Action: Shakespeare Sonnets QA System
Action Input: How does Shakespeare describe love and time in his sonnets?
Observation: In Shakespeare's sonnets, love and time are often portrayed as intertwined forces that influence human experience and beauty. Time is depicted as a relentless and destructive force that leads to aging and decay, threatening to diminish the beauty and vitality of youth. For instance, in Sonnet 16, Shakespeare reflects on how time debates with decay, suggesting that love must contend with the inevitable passage of time.

Conversely, love is portrayed as a powerful and enduring emotion that can transcend the effects of time. In Sonnet 60, Shakespeare compares the growth of men to plants, emphasizing that while both are subject to the same sky (or

{'input': 'How does Shakespeare describe love and time in his sonnets?',
 'output': "In Shakespeare's sonnets, love and time are often portrayed as intertwined forces that influence human experience and beauty. Time is depicted as a relentless and destructive force that leads to aging and decay, threatening to diminish the beauty and vitality of youth. For instance, in Sonnet 16, Shakespeare reflects on how time debates with decay, suggesting that love must contend with the inevitable passage of time.\n\nConversely, love is portrayed as a powerful and enduring emotion that can transcend the effects of time. In Sonnet 60, Shakespeare compares the growth of men to plants, emphasizing that while both are subject to the same sky (or fate), love can serve as a means to preserve beauty and memory against time's ravages. In Sonnet 18, he famously asserts that his beloved's beauty will live on through his poetry, suggesting that art can immortalize love and beauty, defying time's destructive n

In [23]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
I need to compare the features and benefits of ruff and flake8 to provide a comprehensive answer about why one might choose ruff over flake8. 
Action: Ruff QA System
Action Input: Why should someone use ruff over flake8?
Observation: Someone might choose to use Ruff over Flake8 for several reasons:

1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.

2. **Comprehensive Rule Set**: Ruff implements over 800 rules, including many popular Flake8 plugins re-implemented natively, which can cover a broader range of linting issues.

3. **Automatic Fixes**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.

4. **No Custom Rules**: Ruff does not support custom lint rules, which simplifies the setup and usage compared to Flake8, which allows for plugins and custom rules.

5. **Compatibility with Formatters**: Ruff is designed to work well alongside forma

{'input': 'Why use ruff over flake8?',
 'output': 'Someone might choose to use Ruff over Flake8 for several reasons:\n\n1. **Performance**: Ruff is designed to be faster than Flake8, providing quicker feedback on lint violations.\n\n2. **Comprehensive Rule Set**: Ruff implements over 800 rules, including many popular Flake8 plugins re-implemented natively, which can cover a broader range of linting issues.\n\n3. **Automatic Fixes**: Ruff is capable of automatically fixing its own lint violations, which Flake8 does not support.\n\n4. **No Custom Rules**: Ruff does not support custom lint rules, which simplifies the setup and usage compared to Flake8, which allows for plugins and custom rules.\n\n5. **Compatibility with Formatters**: Ruff is designed to work well alongside formatters like Black, and it can defer stylistic rules that are handled by automated formatting.\n\n6. **Drop-in Replacement**: Ruff can serve as a drop-in replacement for Flake8 under certain conditions, making it ea

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [24]:
tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description="useful for when you need to answer questions about Shakespeare's sonnets. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [25]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [26]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Does Shakespeare mention anything related to careful checking or review in his sonnets?"
)



> Entering new AgentExecutor chain...
I need to find out what tool Ruff uses to run over Jupyter Notebooks and also check if there are any references in Shakespeare's sonnets about careful checking or review. 

Action: Ruff QA System  
Action Input: What tool does Ruff use to run over Jupyter Notebooks?  
Observation: Ruff integrates with nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought:I have found the information about Ruff and its integration with Jupyter Notebooks. Now, I need to check if there are any references in Shakespeare's sonnets about careful checking or review.

Action: Shakespeare Sonnets QA System  
Action Input: Does Shakespeare mention anything related to careful checking or review in his sonnets?  
Observation: Yes, Shakespeare does mention careful checking or review in his sonnets. In Sonnet XLVIII, he reflects on how careful he was in safeguarding his "jewels" (which can be interpreted as his thoughts or feelings) to protect th

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Does Shakespeare mention anything related to careful checking or review in his sonnets?',
 'output': "Ruff uses nbQA to run over Jupyter Notebooks. In Shakespeare's sonnets, particularly Sonnet XLVIII, he reflects on the theme of careful checking and review, expressing concern about safeguarding his thoughts and feelings from falsehood and theft."}